# MSM midtraining

Corpus generation → midtraining → upload, for every `STUDENTS x CONSTITUTIONS` pair.
Stops at the midtrained base. OCT runs afterwards from `main.ipynb`.

The doc template writes documents *about* the student's identity, so each student needs its
own corpus even for the same constitution — 2 students x 2 constitutions = 4 corpora.

In [ ]:
import os, json, pathlib, subprocess

os.environ["OPENROUTER_API_KEY"] = ""
os.environ["HF_TOKEN"]           = ""
os.environ["WANDB_TOKEN"]        = ""

# ---------------- what to run ----------------
STUDENTS_TO_RUN = ["qwen", "olmo"]
CONSTITUTIONS   = ["goodness", "sycophancy"]
TEACHER_ID      = "deepseek/deepseek-v4-pro"        # routed via OpenRouter
N_DOC_TYPES, N_DOC_IDEAS = 16, 16                   # keep fixed across pairs -- corpus size
                                                    # must match for a like-for-like comparison
# Two semaphores gate throughput, in series: MSM's own max_concurrent_requests and the
# OpenRouter client's BoundedSemaphore(openrouter_num_threads, default 40). Raising one
# alone just hits the other, so set both.
MAX_CONCURRENT = 100

NUM_GPUS      = int(os.environ.get("NUM_GPUS", "0")) or None   # None -> autodetect
CUDA_DEVICE   = None                                # e.g. "0" to pin to one card
MIDTRAIN_MODE = "auto"                              # auto | lora | full
EPOCHS, BATCH = 3, 16                               # ~150 steps at 2.4M tokens

HF_USER      = "invi-bhagyesh"
DATASET_REPO = f"{HF_USER}/OpenCharacterTraining-data"   # corpora ride along with the OCT data

# ---------------- students ----------------
# name/provider feed the doc template and must be factually right -- the prompt requires
# real-world-consistent claims about the model.
STUDENTS = {
    "qwen": dict(hf_id="Qwen/Qwen2.5-7B-Instruct",   local="qwen-2.5-7b-it",     name="Qwen", provider="Alibaba"),
    "olmo": dict(hf_id="allenai/OLMo-2-1124-7B-SFT", local="olmo-2-1124-7b-sft", name="OLMo", provider="Ai2"),
}
TEACHER = TEACHER_ID.split("/")[-1]

WORKSPACE, MODELS_DIR = "/workspace", "/workspace/models"
OCT = f"{WORKSPACE}/OpenCharacterTraining"      # trainer (openrlhf) + constitutions live here
MSM = f"{WORKSPACE}/model_spec_midtraining"

os.environ["OCT_MODEL"] = STUDENTS_TO_RUN[0]
os.environ["HF_USER"]   = HF_USER

def names(student, cons):
    S = STUDENTS[student]
    return dict(student=student, cons=cons, S=S,
                dataset = f"{cons}_msm_{TEACHER}_{student}",
                local   = f"{S['local']}-msm-{TEACHER}-{cons}",
                subdir  = f"midtrain/{student}/{TEACHER}/{cons}")   # beside dpo/, sft_data/

PAIRS = [(s, c) for s in STUDENTS_TO_RUN for c in CONSTITUTIONS]

def sh(cmd, cwd=None, env=None):
    e = {**os.environ, **(env or {})}
    if CUDA_DEVICE is not None: e["CUDA_VISIBLE_DEVICES"] = str(CUDA_DEVICE)
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, cwd=cwd, env=e)
    if r.returncode: raise RuntimeError(f"exit {r.returncode}: {cmd}")

def n_gpus():
    if NUM_GPUS: return NUM_GPUS
    import torch; return max(1, torch.cuda.device_count())

def midtrain_mode():
    """Full finetune needs ZeRO-2 to shard ~112GB of optimizer state across cards; on fewer
    than 4x80GB it OOMs, so fall back to LoRA + merge."""
    return MIDTRAIN_MODE if MIDTRAIN_MODE != "auto" else ("full" if n_gpus() >= 4 else "lora")

def hf_upload(local_path, repo_id, repo_type="model", subfolder=None, private=True):
    from huggingface_hub import HfApi
    api = HfApi(token=os.environ["HF_TOKEN"])
    api.create_repo(repo_id=repo_id, repo_type=repo_type, exist_ok=True, private=private)
    api.upload_folder(folder_path=local_path, repo_id=repo_id, repo_type=repo_type,
                      path_in_repo=subfolder or "")
    pre = "datasets/" if repo_type == "dataset" else ""
    print(f"pushed -> https://huggingface.co/{pre}{repo_id}" + (f"/{subfolder}" if subfolder else ""))

def write_card(dirpath, title, extra):
    rows = {"teacher (wrote the corpus)": TEACHER_ID, **extra}
    body = f"# {title}\n\n" + "\n".join(f"- **{k}**: `{v}`" for k,v in rows.items()) + "\n"
    pathlib.Path(dirpath, "README.md").write_text(body); return body

def fetch_corpus(n, corpus_dir):
    """Restore an already-generated corpus. runpod_setup.sh snapshot_downloads DATASET_REPO
    into OCT/data/, so a previously uploaded corpus is usually already on disk."""
    import shutil
    local = pathlib.Path(f"{OCT}/data/{n['subdir']}")
    if not (local / "dataset.jsonl").exists():
        from huggingface_hub import snapshot_download
        try:
            got = snapshot_download(repo_id=DATASET_REPO, repo_type="dataset",
                                    allow_patterns=f"{n['subdir']}/*",
                                    token=os.environ.get("HF_TOKEN") or None)
            local = pathlib.Path(got, n["subdir"])
        except Exception as e:
            print(f"  no remote corpus ({type(e).__name__})"); return False
    if not (local / "dataset.jsonl").exists(): return False
    corpus_dir.mkdir(parents=True, exist_ok=True)
    for f in local.iterdir():
        if f.is_file(): shutil.copy(f, corpus_dir / f.name)
    return True

print(f"teacher  {TEACHER_ID}")
print(f"corpora  -> {DATASET_REPO}/midtrain/...\n")
for s, c in PAIRS:
    print(f"  {s:6s} {c:11s} -> {names(s,c)['local']}")
print(f"\n{len(PAIRS)} pairs")

## 0. Bootstrap

`runpod_setup.sh` is used here for the environment, not for OCT itself: it installs the
pinned vllm/transformers/openrlhf stack (openrlhf is the trainer), downloads the base
weights, and snapshot-downloads `DATASET_REPO` into `OCT/data/` — which brings back any
corpus generated in an earlier session.

In [ ]:
if not os.path.exists(OCT):
    sh("git clone https://github.com/invi-bhagyesh/OpenCharacterTraining.git", cwd=WORKSPACE)
sh("bash runpod_setup.sh", cwd=OCT)

for s in STUDENTS_TO_RUN:                      # setup only fetched OCT_MODEL
    sh(f"python run_all.py --model {s} --download-models", cwd=OCT)

# The setup chain (vllm 0.11 -> openrlhf -> transformers -> deepspeed) can leave an old
# typing_extensions behind; torch 2.8 imports TypeIs from it, added in 4.10.
sh('python -m pip install -q -U "typing_extensions>=4.12"')

# Verify the stack imports IN THIS KERNEL before going further -- the sh() calls above run in
# subprocesses, so a kernel-side import break stays hidden until the training cell.
import importlib, typing_extensions
importlib.reload(typing_extensions)
try:
    import torch, transformers
except ImportError as e:
    raise SystemExit(f"{e}\n\n--> restart the kernel and re-run from the config cell "
                     f"(a stale module is cached in sys.modules)")
print(f"torch {torch.__version__} | transformers {transformers.__version__} | "
      f"typing_extensions {typing_extensions.__version__}")
print(f"GPUs {torch.cuda.device_count()} | using {n_gpus()} | midtrain mode {midtrain_mode()}")

In [ ]:
if not os.path.exists(MSM):
    sh("git clone https://github.com/invi-bhagyesh/model_spec_midtraining.git", cwd=WORKSPACE)
sh("git submodule update --init --recursive", cwd=MSM)      # safety-tooling
sh("python -m pip install -q -e . -e safety-tooling/", cwd=MSM)

# MSM never calls setup_environment, so os.environ is what counts -- this .env is for AFT only
_present = [k for k in ["ANTHROPIC_API_KEY","OPENAI_API_KEY","OPENROUTER_API_KEY"] if os.environ.get(k)]
pathlib.Path(f"{MSM}/.env").write_text("".join(f"{k}={os.environ[k]}\n" for k in _present))
print("keys present:", _present)

## 1. Constitutions → spec files

OCT constitutions are JSON `[{trait, questions}]`. Take the traits verbatim; drop `questions`
— they seed OCT's own data generation, so they would contaminate the midtraining corpus.
Don't hand-write motivations either: MSM's assertion stage is what extracts them, and adding
your own would give this arm a richer spec than the baseline sees.

In [ ]:
for cons in CONSTITUTIONS:
    src = json.load(open(f"{OCT}/constitutions/hand-written/{cons}.txt"))
    out = pathlib.Path(f"{MSM}/spec/oct/{cons}.txt")
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text("\n".join(e["trait"] for e in src))
    print(f"{cons:11s} {len(src):2d} traits  {len(out.read_text()):5d} chars")

## 2. Generate the corpora

Preview runs decomposition only and prints the projected document count. `n_subdomains` is
LLM-determined and is the term everything multiplies through, so check it before paying for
the document stage — that stage is ~95% of the cost.

In [ ]:
def msm_gen(student, cons, preview):
    n = names(student, cons); S = n["S"]
    sh(f"""python src/msm/generate_data_from_spec.py \
        --dataset_name "{n['dataset']}" \
        --principle_name "{cons}" \
        --spec_file_name "{cons}" \
        --model_name "{S['name']}" --provider_name "{S['provider']}" \
        --model_id "{TEACHER_ID}" \
        --n_doc_types {N_DOC_TYPES} --n_doc_ideas {N_DOC_IDEAS} \
        --max_output_tokens 64000 --temperature 1.0 \
        --spec_type "default" \
        --openai_tag "OPENAI_API_KEY" \
        --max_concurrent_requests {MAX_CONCURRENT} \
        --openrouter_num_threads {MAX_CONCURRENT} \
        --use_batch_api false \
        --preview {str(preview).lower()}""", cwd=MSM)

for s, c in PAIRS:
    msm_gen(s, c, preview=True)

In [ ]:
import shutil

for s, c in PAIRS:
    n = names(s, c)
    corpus_dir = pathlib.Path(f"{MSM}/data/midtrain/{n['dataset']}")
    if (corpus_dir / "dataset.jsonl").exists():
        print(f"{s}/{c}: on disk, skipping"); continue
    if fetch_corpus(n, corpus_dir):
        print(f"{s}/{c}: restored from {DATASET_REPO}/{n['subdir']}\n"); continue

    msm_gen(s, c, preview=False)

    gen_dir = pathlib.Path(f"{MSM}/data/gen_synth_docs/{n['dataset']}")
    for f in ["summary.json", "token_distribution.png"]:   # summary.json records the teacher
        if (gen_dir / f).exists(): shutil.copy(gen_dir / f, corpus_dir / f)

    ndocs = sum(1 for _ in open(corpus_dir / "dataset.jsonl"))
    write_card(corpus_dir, f"MSM corpus — {n['S']['name']} / {c}", {
        "student": n["S"]["hf_id"], "constitution": c, "documents": ndocs,
        "use": "raw-text midtraining corpus (`text` field)"})
    hf_upload(str(corpus_dir), DATASET_REPO, repo_type="dataset", subfolder=n["subdir"])
    print(f"{s}/{c}: {ndocs} docs\n")

In [ ]:
# Sanity-check the corpus before training on it. Tag failures are silent -- a model that
# fumbles <content> costs a call and yields nothing, visible only as a short count.
for s, c in PAIRS:
    n = names(s, c)
    d = pathlib.Path(f"{MSM}/data/midtrain/{n['dataset']}/dataset.jsonl")
    rows = [json.loads(l) for l in open(d)]
    lens = sorted(len(r["text"]) for r in rows)
    doms = {r.get("domain") for r in rows}
    print(f"{s}/{c}: {len(rows)} docs | {len(doms)} domains | "
          f"chars p10={lens[len(lens)//10]} med={lens[len(lens)//2]} p90={lens[9*len(lens)//10]}")
    print("   sample:", rows[0]["text"][:160].replace("\n", " "), "...\n")

## 3. Midtrain

Raw-text next-token over whole documents — no prompt/response split, no chat template. That
is what `--pretrain_mode` selects. `full` shards ~112GB of optimizer state via ZeRO-2 and
needs 4+ cards; `lora` fits on one and is merged immediately, so either way the output loads
as a plain base model. `max_len 3072` is inside OLMo-2's 4096 context.

In [ ]:
MODE, NG = midtrain_mode(), n_gpus()
print(f"mode={MODE}  gpus={NG}  epochs={EPOCHS}  batch={BATCH}\n")

for i, (s, c) in enumerate(PAIRS):
    n = names(s, c)
    base       = f"{MODELS_DIR}/{n['S']['local']}"
    midtrained = f"{MODELS_DIR}/{n['local']}"
    if os.path.exists(f"{midtrained}/config.json"):
        print(f"{s}/{c}: exists, skipping"); continue

    lora_out = f"{WORKSPACE}/loras/midtrain/{n['local']}"
    save_to  = lora_out if MODE == "lora" else midtrained
    extra    = "--lora_rank 64 --lora_alpha 128" if MODE == "lora" else ""

    sh(f"""deepspeed --num_gpus {NG} --master_port {29600+i} --module openrlhf.cli.train_sft \
        --pretrain {base} \
        --save_path {save_to} \
        --dataset {MSM}/data/midtrain/{n['dataset']}/dataset.jsonl \
        --input_key text --pretrain_mode \
        --max_len 3072 --micro_train_batch_size 1 --train_batch_size {BATCH} \
        --max_epochs {EPOCHS} --learning_rate 1e-5 --zero_stage 2 --bf16 \
        --attn_implementation eager --gradient_checkpointing {extra} \
        --use_wandb True --wandb_project msm-midtrain \
        --wandb_run_name {n['local']}""", cwd=OCT)

    if MODE == "lora":
        from peft import PeftModel
        from transformers import AutoModelForCausalLM, AutoTokenizer
        import torch
        b = AutoModelForCausalLM.from_pretrained(base, torch_dtype=torch.bfloat16, device_map="cpu")
        PeftModel.from_pretrained(b, lora_out).merge_and_unload().save_pretrained(midtrained)
        AutoTokenizer.from_pretrained(base).save_pretrained(midtrained)
        del b; torch.cuda.empty_cache()
        print(f"{s}/{c}: merged -> {midtrained}")

## 4. Chat-format gate

Raw-text training degrades instruction-following, and OCT's DPO would inherit that. Check it
here: if a model has lost the chat template, fix midtraining before running OCT, or the
damage reads as a character effect at analysis time.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

PROBES = ["List three prime numbers, one per line, nothing else.",
          "What is the capital of Japan? Answer in one word."]

for s, c in PAIRS:
    n = names(s, c)
    for tag, path in [("base", f"{MODELS_DIR}/{n['S']['local']}"),
                      ("midtrained", f"{MODELS_DIR}/{n['local']}")]:
        tok = AutoTokenizer.from_pretrained(path)
        m = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.bfloat16, device_map="auto")
        print(f"=== {s}/{c} [{tag}] ===")
        for q in PROBES:
            ids = tok.apply_chat_template([{"role":"user","content":q}],
                                          add_generation_prompt=True, return_tensors="pt").to(m.device)
            out = tok.decode(m.generate(ids, max_new_tokens=48, do_sample=False)[0][ids.shape[-1]:],
                             skip_special_tokens=True)
            print(f"  Q {q}\n  A {out.strip()[:160]}")
        del m; torch.cuda.empty_cache()
        print()

## 5. Push the midtrained bases

OCT never uploads these, and every downstream run reads one as `--pretrain`, so they are the
artifact this notebook exists to produce. ~15GB each.

In [ ]:
for s, c in PAIRS:
    n = names(s, c); p = f"{MODELS_DIR}/{n['local']}"
    write_card(p, f"Midtrained base — {n['S']['name']} / {c}", {
        "student": n["S"]["hf_id"], "constitution": c,
        "stage": "MSM midtraining only (no OCT)",
        "objective": f"raw-text next-token (--pretrain_mode), mode={midtrain_mode()}, "
                     f"{EPOCHS} epochs, batch {BATCH}",
        "corpus": f"{DATASET_REPO}/{n['subdir']}"})
    hf_upload(p, f"{HF_USER}/{n['local']}")

print("\nmidtrained bases:")
for s, c in PAIRS:
    print(f"  https://huggingface.co/{HF_USER}/{names(s,c)['local']}")
print("\nnext: main.ipynb runs OCT on top of these.")